# Modelling a glacier with IGM — tutorial (Sessions 2 and 3)

This tutorial carries out a series of **numerical experiments with IGM** on the **[Rhone Glacier](https://en.wikipedia.org/wiki/Rhone_Glacier)** (Valais, ~15 km², the source of the Rhone river; RGI ID `RGI2000-v7.0-G-11-01706`, set by default in all parameter files). Step by step, you go through the **basic workflow** of a glacier modelling study: check the data, calibrate the ice flow on the observed speeds, calibrate the melt on the past evolution, project the glacier until 2100, and retrieve the ELA history that explains its evolution.

It is the **basis of your mini-project**: once you have gone through it with the Rhone Glacier, **migrate to your own glacier** (put its RGI ID in each file of `experiment/`) and redo the same steps, still in Session 2. Session 3 then deepens some experiments on your glacier (LIA extent, response time, moraines): **second part of this notebook** (sections 6–8).

**Follow this notebook from top to bottom.** It lives in **your own copy** of the template, `~/work/my-glacier/` (copied once from `~/work/glamod/session2/my-glacier/`). It only **reads and plots** the results of your IGM runs: re-run it as often as you like.

| What | Where |
|---|---|
| **Set** the parameters | YAML files in `experiment/` (text editor) |
| **Run** IGM | a **Terminal**, from this folder: `cd ~/work/my-glacier` |
| **Look** at the results | this notebook, reading `outputs/<run>/` |

**Session 2 — the basic workflow** (sections 1–5):
1. **Present**: download the data, check the initial state
2. **Calibration of the ice-flow parameters** (`arrhenius`, `tau_ref`) on the observed speeds
3. **Past**: Little Ice Age → today, **calibration of the SMB** (`melt_enhancer`)
4. **Future**: climate scenarios 2020–2100
5. **Retrieving the ELA history**: a simple (ELA-based) SMB model fitted to the complex one

**Session 3 — experiment deepening** (sections 6–8, further down, after the line *Session 3*):

6. **The Little Ice Age extent** (**everyone**): a closer calibration of the past
7. **Extension A — the response time**: your own IGM module
8. **Extension B — the moraines**: debris particles

In [ ]:
import sys, os, re, yaml
sys.path.insert(0, os.path.expanduser("~/work/glamod/utils"))   # course plotting helpers (utils/helper.py)
sys.path.insert(0, os.path.abspath("../../utils"))               # (same, when run inside the course folder)
from helper import *

with open("experiment/params_past.yaml") as f:
    RGI_ID = yaml.safe_load(f)["inputs"]["oggm_shop"]["RGI_ID"]
INPUT = f"data/input_{RGI_ID}.nc"
print("My glacier:", RGI_ID)
os.makedirs("figures", exist_ok=True)                  # figures for the report go here

## 1. Present: download the data and check the initial state

**What happens here**: `oggm_shop` downloads everything IGM needs for the glacier: the topography, the glacier outline, an estimate of the ice thickness (Millan et al., 2022), the observed surface speeds and the climate (1901–2019). Before any experiment, we check that these data make sense (thickness, speeds, climate trend), since the whole project relies on them.

▶ **In the terminal**: a 1-year run downloads the data (≈ 1 min the first time):

```bash
igm_run +experiment=params_present
```

In [ ]:
plot_input_data(INPUT)
plot_climate(RGI_ID)

## 2. Calibration of the ice-flow parameters

**Why?** The ice flow depends on two poorly known parameters: the ice softness (`arrhenius`, deformation) and the basal sliding (`tau_ref`). The modelled speeds must match the **observed** ones, otherwise the glacier moves too much or too little ice downstream, and its evolution is wrong. We calibrate these two parameters **once, on today's glacier**, and then keep them for all other experiments.

First, compare the speeds modelled with the default parameters (`arrhenius` = 78, `tau_ref` = 0.215) with the observed ones (Millan et al., 2022):

In [ ]:
compare_modeled_observed_velocity(INPUT, "outputs/params_present/output.nc", year=2020)

Then explore **all combinations** of ice softness (`arrhenius`) and basal sliding (`tau_ref`) with 1-year runs. `-m` (multirun) makes one run per combination of the values given in the command line (3 × 4 = 12 runs, ≈ 3 min).

▶ **In the terminal**:

```bash
igm_run -m +experiment=params_sensitivity processes.iceflow.physics.viscosity.arrhenius=25,50,100 processes.iceflow.physics.sliding.tau_ref=0.1,0.2,0.4,0.8
```

Results: `outputs/params_sensitivity/0`, `1`, …, `11`. Each map below shows the speed difference (modelled − observed) of one run: look for a **bias close to 0** and a small RMSE (in red). Refine the values around the best one if needed (re-run with other values: the folder is overwritten).

In [ ]:
runs, labels = sweep_runs("outputs/params_sensitivity")   # the 12 runs
compare_velocity_misfit(INPUT, runs, year=2020)

> ✅ **Calibration 1 — ice flow** (double-click to edit): `arrhenius` = ...  `tau_ref` = ...
>
> (example, Rhone Glacier: `arrhenius` = 30, `tau_ref` = 0.315)
>
> ➡️ Put these values in the `iceflow → physics` block of **all** the files of `experiment/`.

## 3. Past: from the Little Ice Age to today — calibration of the SMB

**Why?** The SMB computed from the climate (temperature, precipitation) is uncertain. We calibrate it by checking that the model reproduces the **known history** of the glacier: an advance until the Little Ice Age maximum (~1850, marked by moraines), then the retreat until today. If the model gets the past right, we can trust it more for the future.

**Experiment**: we do not know the glacier geometry in 1700, so we start from **today's** geometry with a cooler pre-industrial climate: the glacier grows to its LIA maximum, then retreats with the observed climate 1901–2019. A single parameter is calibrated: `melt_enhancer` (more or less melt).

Start in 1700 from today's geometry, advance to the LIA maximum (~1850), retreat with the observed climate 1901–2019 (≈ a few minutes). Keep the pre-1901 climate (`clim_trend_array`) of `experiment/params_past.yaml` and **adjust `melt_enhancer`** (start: 0.8, explore 0.5–2) until the glacier (i) reaches the LIA moraines around 1850 (Siegfried map on map.geo.admin.ch) and (ii) matches today's ice volume in 2020.

▶ **In the terminal**:

```bash
igm_run +experiment=params_past
```

To keep several tries, copy the file (e.g. `params_past_v2.yaml`, and change its run folder at the end of the file to `outputs/params_past_v2`) and add the run to `past_runs` below.

In [ ]:
run = "outputs/params_past"
show_dashboard(run, n=4)
analyze_time_series(os.path.join(run, "output_ts.nc"))

In [ ]:
# Ice thickness through time, and along the main flowline
animate_glacier_evolution(os.path.join(run, "output.nc"), variable="thk")
plot_flowline_evolution(os.path.join(run, "output.nc"), rgi_id=RGI_ID)

In [ ]:
# Compare the volume of several past runs with today's volume (ice thickness of the input data)
ds = xr.open_dataset(INPUT); dx = float(ds.x[1] - ds.x[0])
obs = {"vol": ([2020], [float(ds.thk.sum()) * dx**2 / 1e9])}
past_runs = ["outputs/params_past"]              # <- add your other past runs here
compare_runs(past_runs, [os.path.basename(r) for r in past_runs], observations=obs)

> ✅ **Calibration 2 — SMB** (double-click to edit): `melt_enhancer` = ...
>
> ➡️ Put this value in `params_future.yaml` (and `params_past_moraines.yaml` for Session 3).

## 4. Future: climate scenarios 2020–2100

**Why?** With the calibrated ice flow (section 2) and SMB (section 3), we project the glacier until 2100 under several **climate scenarios** (from "committed", i.e. today's climate kept constant, to RCP8.5, strong warming). The comparison tells how much of the glacier is already lost whatever we do, and how much depends on future emissions.

`delta_temp` is relative to **1960–1990** (see Lecture 2):
- 2020 row (in `experiment/params_future.yaml`): present-day offset (2000–2019, printed by `plot_climate` above)
- 2100: 1981–2010 offset + CH2018 warming of the scenario, given **in the command line**, one value per scenario (committed = today's offset, RCP2.6, RCP4.5, RCP8.5)

▶ Check that `params_future.yaml` contains your calibrations 1 and 2, then **in the terminal** (4 runs, a few minutes; values here for the Rhone Glacier, adapt them):

```bash
igm_run -m +experiment=params_future processes.climate.oggm.clim_trend_array.2.1=1.3,2.0,3.4,5.1
```

One run per scenario: `outputs/params_future/0`, `1`, `2`, `3` (in the order of the list).

In [ ]:
runs, labels = sweep_runs("outputs/params_future")    # the scenario runs, labelled by their 2100 delta_temp
compare_runs(runs, labels)

In [ ]:
# Maps for one scenario (here the third one: RCP4.5)
animate_glacier_evolution(runs[2] + "/output.nc", variable="thk")

## 5. Retrieving the ELA history: a simple SMB model fitted to the complex one

**Why?** The climate-driven SMB hides many processes (temperature, precipitation, melt factors). Here we take a simpler approach: we prescribe the **ELA directly**. This forces us to think about **which ELAs must have prevailed** to drive the glacier evolution we observe (LIA advance, then retreat).

**Goal**: try to reproduce the volume curve of your calibrated past run (section 3) by playing with the ELAs (and gradients) in `experiment/params_past_simple.yaml` (today's ELA: end-of-summer snowline on map.geo.admin.ch, GLAMOS; LIA ELAs ≈ 100–200 m lower).

**Difficulty**: a glacier does not react immediately. It has a **response time**: after an ELA change, it takes years to decades to adjust its geometry. So the ELA must change *before* the volume does. This is studied in Session 3 (section 7).

▶ **In the terminal**:

```bash
igm_run +experiment=params_past_simple
```

In [ ]:
plot_smb_simple(ela=3000, gradabl=0.009, gradacc=0.005, accmax=2.0)   # <- your ELA and gradients
compare_runs(["outputs/params_past", "outputs/params_past_simple"], ["climate-driven SMB", "ELA-based SMB"], observations=obs)

---
---

# Session 3: experiment deepening

**⚠️ Session 2 ends here. Sections 6–8 are for Session 3.**

In Session 2 you went through the **basic workflow** (sections 1–5) with the Rhone Glacier, then with your own glacier. Session 3 **deepens** some experiments on your glacier, by bringing new data and by customising IGM:

6. **The Little Ice Age extent** (**everyone**): compare the modelled glacier in 1850 with the mapped 1850 extent, and refine your calibration of the past
7. **Extension A — the response time**: write / read your own IGM module (oscillating ELA)
8. **Extension B — the moraines**: track debris with particles (if relevant for your glacier)

Section 6 (fine-tuning the calibration on the LIA extent) is **necessary for everyone**. Sections 7–8 are **extensions**, to be taken afterwards.

Keep working in **your** project `~/work/my-glacier` (from Session 2): do **not** copy the template again, your runs in `outputs/` and your YAML files stay as they are. If your copy of this notebook stops at section 5, get the updated course material and the updated notebook, in a terminal (the `mv` keeps your Session 2 notebook, with your notes, under another name):

```bash
cd ~/work/glamod && git pull
cd ~/work/my-glacier
mv tutorial.ipynb tutorial-session2.ipynb
cp ~/work/glamod/session2/my-glacier/tutorial.ipynb .
```

Then re-run the first code cell (setup) of this notebook before the cells below.

## 6. The Little Ice Age extent: a closer calibration of the past

**Why?** In Session 2 you calibrated the melt (`melt_enhancer`) mostly on today's volume, and checked the LIA maximum by eye. The LIA maximum (~1850) is the best-documented past state of Alpine glaciers: its **moraines** are still visible, and it was mapped on the first topographic maps. Comparing the modelled extent in 1850 with the mapped one is a strong test of the model, and helps to constrain the **past climate** (pre-1901 `delta_temp`) and the **melt**.

**Resources for the 1850 extent**:
- **Swiss Glacier Inventory 1850** (SGI 1850, [GLAMOS](https://www.glamos.ch/en/downloads#inventories); Maisch et al., 2000): outlines of all Swiss glaciers in 1850, from historical maps and moraines. Downloaded automatically below (`data/sgi1850/`).
- **Historical maps** on [map.geo.admin.ch](https://map.geo.admin.ch) (*Journey through time*): Dufour map (1845–1865) and Siegfried map (1870–1926), close to the LIA maximum.
- **Moraines** on the relief (*swissALTI3D* hillshade) and aerial images (*SWISSIMAGE*) of map.geo.admin.ch: the LIA moraines are the youngest, fresh, unvegetated ridges in front of the glacier.
- **Length changes** since ~1880 (GLAMOS factsheet of your glacier, e.g. [Rhone Glacier](https://www.glamos.ch/en/factsheet#/B43-03)): the retreat of the front.

▶ First compare your calibrated past run (Session 2) with the 1850 outline:

In [ ]:
run = "outputs/params_past"                         # <- your calibrated past run (Session 2)
compare_lia_extent(run, INPUT, year=1850)
compare_runs([run])

**Then refine the calibration of the past**: copy `experiment/params_past.yaml` (e.g. `params_past_lia.yaml`, with the run folder `outputs/params_past_lia` at the end of the file), and adjust one parameter at a time:
- the glacier is **too small / too large in 1850** ➡️ colder / warmer pre-1901 `delta_temp` (rows 1700 and 1850 of `clim_trend_array`)
- the glacier is **too small / too large today** ➡️ more / less melt: `melt_enhancer`
- the front is too thin and advances too slowly / too fast ➡️ check the ice flow again (`arrhenius`, `tau_ref`, section 2)

```bash
igm_run +experiment=params_past_lia
```

Aim for a run that reaches the 1850 outline **and** today's volume; no perfect fit is possible, **discuss** the remaining mismatch in your report. If your calibration changes, redo the future scenarios (`params_future.yaml`).

In [ ]:
run = "outputs/params_past_lia"
compare_lia_extent(run, INPUT, year=1850)
compare_runs(["outputs/params_past", run])

> ✅ **Calibration of the past, refined** (double-click to edit): pre-1901 `delta_temp` = ...  `melt_enhancer` = ...  (`arrhenius` = ...  `tau_ref` = ...)

## 7. Extension A — Response time: your own IGM module

**Why?** In section 5 you saw that the glacier does not react immediately to an ELA change. How long does it take, and what does it mean for its future?

**The experiment**: the ELA oscillates like a sine wave (period $P$ = 200 yr, amplitude 200 m) and we look at how the glacier volume follows. This SMB is not part of IGM: it is a **user module**, `user/code/processes/smb_oscillating.py` (parameters: `user/conf/processes/smb_oscillating.yaml`). Open it to see how you can extend IGM with your own processes.

▶ **In the terminal**:

```bash
igm_run +experiment=params_response_time
```

In [ ]:
plot_response_time("outputs/params_response_time")

**Your turn**:

1. **Find the response time** $\tau$ of your glacier from the lag between the ELA and the volume (first-order model, Lecture 3).
2. **Test other periods** $P$ (copy the YAML file and adapt it): how do the lag and the amplitude of the volume change? Which periods give a reliable $\tau$? Compare with the theoretical estimate $\tau \approx H / (-b_{\text{terminus}})$ (Lecture 1a).
3. **Size**: how does $\tau$ relate to the thickness and volume of your glacier? Compare with other students' glaciers.
4. **Conclude**: how does your glacier respond to short climate variations? to long-term trends? What does it mean for 2100?

> ✅ **Conclusion — response time** (double-click to edit): ...

## 8. Extension B — Moraines: reproduce the LIA moraines

**Why?** Rocks fallen on the glacier are buried, carried by the ice flow and released at the margins and the front, where they build **lateral and frontal moraines**. The LIA moraines are the youngest, fresh ridges in front of today's glaciers. Can the model reproduce **where** they are? IGM tracks the debris with **particles** moved by the 3D ice flow: when the glacier retreats after ~1850, the particles left on the ice-free terrain show where the modelled glacier dropped its debris.

▶ **In the terminal**: put your calibrations (ice flow, `melt_enhancer`, pre-1901 climate from section 6) in `experiment/params_past_moraines.yaml`, then (a few minutes, more than the past run)

```bash
igm_run +experiment=params_past_moraines
```

In [ ]:
run_p = "outputs/params_past_moraines"
plot_particles(run_p, year=1850)                    # debris in the glacier at the LIA maximum (colour: depth)
plot_moraines(run_p, INPUT, year=2020)              # debris left on ice-free terrain today = modelled LIA moraines

**Compare with the real moraines**: on [map.geo.admin.ch](https://map.geo.admin.ch), look at the aerial images (SWISSIMAGE) and the relief (swissALTI3D hillshade) around your glacier: where are the LIA lateral and frontal moraines (fresh, unvegetated ridges)? Are the modelled deposits at the same place, at the same height on the valley sides?

**Play with the controls** (`particles: seeding:` in `params_past_moraines.yaml`; copy the file and change its run folder for each try):
- `method`: `accumulation` (surface of the accumulation area, regularly) or `all` (the whole ice column, once at the start)
- `frequency` (years between two seedings) and `density` (fraction of grid cells seeded): how much debris
- your calibration of the past (section 6): a glacier too large / too small in 1850 puts its moraines too far / too close

**Discuss the limits**: real debris comes mostly from the **rock walls** above the glacier (rockfall), not from the whole accumulation area; the moraine ridges are built by processes not in the model (dumping, pushing, reworking).